# Phase 3 — Modellierung & Vergleich

AI-Kennzeichnung: AI wurde verwendet, um physische Logiken des Projekts zu verstehen, bewusste architektonische Entscheidungen zu unterstützen, Code-Qualität/Struktur sowie Kommentare zu verbessern.

## 1. Import und Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal as sig
from pathlib import Path
from typing import Callable

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    f1_score, recall_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report, make_scorer,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from tensorflow import keras
from tensorflow.keras import layers

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 10)

In [ ]:
FEATURES_PATH = Path("features_phase2.csv")
DATA_PATH     = Path("condition+monitoring+of+hydraulic+systems")

CLASS_TO_LABEL = {90: 0, 100: 1, 115: 2, 130: 3}
LABEL_TO_CLASS = {v: k for k, v in CLASS_TO_LABEL.items()}
CLASSES        = [90, 100, 115, 130]
CLASS_NAMES    = [f"{c} bar" for c in CLASSES]
CLASS_COLORS   = {90: "#E24B4A", 100: "#EF9F27", 115: "#1D9E75", 130: "#185FA5"}

N_SPLITS     = 3
RANDOM_STATE = 42

SENSORS_100HZ = ["PS1", "PS2", "PS3", "PS4", "PS5", "PS6", "EPS1"]
SENSORS_10HZ  = ["FS1", "FS2"]
SENSORS_1HZ   = ["TS1", "TS2", "TS3", "TS4", "VS1", "CE", "CP", "SE"]
SENSORS_ALL   = SENSORS_100HZ + SENSORS_10HZ + SENSORS_1HZ

FREQUENCY_KEYWORDS = ["fft_coefficient", "fft_aggregated", "spkt_welch_density"]

IQR_FACTOR        = 1.5
LOWPASS_CUTOFF_HZ = 10.0
LOWPASS_ORDER     = 5
DOWNSAMPLE_FACTOR = 10
TARGET_TIMESTEPS  = 600

KERAS_EPOCHS      = 30
KERAS_PATIENCE    = 6
KERAS_LR_PATIENCE = 3
KERAS_LR_FACTOR   = 0.5
KERAS_BATCH_SIZE  = 32

SCORING = {
    "f1_macro": make_scorer(
        f1_score, average="macro", labels=list(CLASS_TO_LABEL.values()), zero_division=0,
    ),
    "balanced_accuracy": make_scorer(
        recall_score, average="macro", labels=list(CLASS_TO_LABEL.values()), zero_division=0,
    ),
}

## 2. Datengrundlage: Merkmale aus Phase 2

`features_phase2.csv` enthält die in Phase 2 extrahierten und selektierten Merkmale aus allen 17 Sensoren, zusammen mit der Zielgröße `Accumulator` und dem `Stable`-Flag.

In [ ]:
def load_features() -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Load Phase 2 feature matrix. Returns (X, y_encoded, stable_flags)."""
    df       = pd.read_csv(FEATURES_PATH, index_col=0)
    df.index = df.index.astype(int)
    y_raw    = df["Accumulator"].values
    stable   = df["Stable"].values
    X        = df.drop(columns=["Accumulator", "Stable"])
    y        = np.array([CLASS_TO_LABEL[v] for v in y_raw])
    return X, y, stable

In [ ]:
X_features, y, stable = load_features()
print(f"Shape: {X_features.shape}")
print(f"Klassen: {dict(zip(*np.unique(y, return_counts=True)))}")

### 2.1 Behandlung ungültiger Werte

Die in Phase 2 exportierte Feature-Matrix wird auf ungültige Werte (NaN, ±inf) geprüft. tsfresh kann für einzelne Frequenzmerkmale inf-Werte erzeugen (z.B. bei Division durch Null); gefundene ungültige Werte werden durch den Spaltenmedian ersetzt. Dieser Schritt deckt den in der Aufgabenstellung für Phase 3 genannten Punkt "Behandlung ungültiger Werte" auf der finalen Feature-Matrix ab.

In [ ]:
def handle_invalid_values(X: pd.DataFrame) -> pd.DataFrame:
    """Replace +/-inf with NaN and impute remaining NaNs with column medians."""
    n_inf = np.isinf(X.to_numpy()).sum()
    X_clean = X.replace([np.inf, -np.inf], np.nan)
    n_nan = X_clean.isna().sum().sum()
    if n_nan > 0:
        X_clean = X_clean.fillna(X_clean.median())
    print(f"Ersetzte inf-Werte: {n_inf}")
    print(f"Imputierte NaN-Werte (nach inf-Ersetzung): {n_nan}")
    return X_clean


X_features = handle_invalid_values(X_features)

### 2.2 Zielgrößen-Encoding

Die Klassen 90/100/115/130 bar sind nicht-kontinuierlich und beginnen nicht bei 0. XGBoost und Keras (`categorical_crossentropy`) erwarten jedoch ganzzahlige Labels im Bereich `[0, n_classes)`. `CLASS_TO_LABEL` bildet dies explizit ab, statt `LabelEncoder` zu verwenden — die Zuordnung bleibt damit nachvollziehbar und konstant.

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(CLASS_NAMES, class_counts.values, color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white")
ax.set_title("Klassenverteilung — Hydraulic Accumulator")
ax.set_ylabel("Anzahl Zyklen")
plt.tight_layout()
plt.show()

## 3. Validierungsstrategie

**Begründung:**

Phase 1 hat festgestellt, dass die Klassen in zeitlichen **Blöcken** statt zufällig aufgezeichnet wurden, und ein Sensor (TS1) stark mit der Experimentreihenfolge korreliert (r ≈ -0.90). Eine rein zufällige Aufteilung könnte diese zeitliche Struktur leaken und die Modellgüte überschätzen.

Daher verwenden wir zwei sich ergänzende Validierungsschemata:
- **StratifiedKFold (3-fold)**: erfüllt die Mindestanforderung von ≥3 Trainingsläufen   pro Modell, Klassenverhältnis bleibt in jedem Fold erhalten.
- **GroupKFold (3-fold)**: Zyklen desselben zeitlichen Blocks landen nie gleichzeitig   in Training und Test — Robustheits-Check gegenüber der Phase-1-Drift-Beobachtung.

Beide Schemata werden für Ansatz 1 (sklearn, geringe Rechenzeit) vollständig mit 3 Folds durchgeführt. Für Ansatz 2 (Keras, hohe Rechenzeit pro Lauf) dient StratifiedKFold als Hauptergebnis (erfüllt die ≥3-Trainingsläufe-Anforderung); GroupKFold wird ebenfalls mit 3 Folds als zusätzlicher Robustheits-Check durchgeführt.

### 3.1 Block-Gruppen für blockbewusste Validierung

In [ ]:
def detect_cycle_blocks(y: np.ndarray) -> np.ndarray:
    """Assign a block id to each cycle. A new block starts whenever the class changes."""
    blocks = np.zeros(len(y), dtype=int)
    for i in range(1, len(y)):
        blocks[i] = blocks[i - 1] + (1 if y[i] != y[i - 1] else 0)
    return blocks


groups = detect_cycle_blocks(y)
print(f"Anzahl Zyklen: {len(y)}")
print(f"Anzahl Blöcke: {len(np.unique(groups))}")

### 3.2 Cross-Validation-Schemata

In [ ]:
cv_stratified = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_group      = GroupKFold(n_splits=N_SPLITS)

## 4. Ansatz 1: Manual Feature Extraction

Verwendung der in Phase 2 extrahierten und selektierten tsfresh-Merkmale aus allen 17 Sensoren.

### 4.1 Modellauswahl & Begründung

**Baseline:**
- `DummyClassifier` (strategy="most_frequent"): Referenzwert — jedes Modell   muss diesen übertreffen.
- `LogisticRegression`: einfaches lineares Baseline-Modell.

**Hauptmodelle (4, davon 1 klassisch, 2 Ensemble, 1 neuronales Netz):**
- `SVC` (klassisches Verfahren, Pflicht): Kernel-Trick erlaubt nichtlineare   Entscheidungsgrenzen, profitiert direkt vom StandardScaler.
- `RandomForest` und `XGBoost` (Ensemble, 2 statt 1): Ensemble-Methoden   übertreffen auf tabellarischen Daten im Mittel neuronale Netze (Grinsztajn   et al. 2022). Da unsere Phase-2-Features tabellarisch sind, decken wir   beide gängigen Ensemble-Typen ab (Bagging vs. Boosting).
- `MLP` (neuronales Netz, Pflicht): `sklearn.neural_network.MLPClassifier` als   tabellarisches Pendant zu den Keras-Modellen in Ansatz 2.

`class_weight="balanced"` wird bei allen Modellen verwendet, die dies unterstützen (LogReg, SVC, RandomForest) — zusätzlich zur Wahl von F1-Macro/Balanced Accuracy als Metriken, um der Klassenimbalance auch beim Training zu begegnen. XGBoost und MLP unterstützen `class_weight` nicht direkt; hier wird ausschließlich über die Metriken bewertet.

In [ ]:
MODELS_ANSATZ1: dict[str, object] = {
    "Dummy":        DummyClassifier(strategy="most_frequent"),
    "LogReg":       LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "SVC":          SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
    "XGBoost":      XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                   random_state=RANDOM_STATE, eval_metric="mlogloss"),
    "MLP":          MLPClassifier(hidden_layer_sizes=(100,), max_iter=500,
                                   early_stopping=True, random_state=RANDOM_STATE),
}

### 4.2 Training & Evaluation

In [ ]:
def build_pipeline(model: object) -> Pipeline:
    """Wrap a classifier in a StandardScaler pipeline (fit on train folds only)."""
    return Pipeline([("scaler", StandardScaler()), ("model", model)])


def evaluate_model(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
) -> pd.DataFrame:
    """Run StratifiedKFold and GroupKFold cross-validation. Returns long-format results."""
    rows = []
    for scheme_name, cv, cv_groups in [
        ("StratifiedKFold", cv_stratified, None),
        ("GroupKFold",      cv_group,      groups),
    ]:
        scores = cross_validate(pipeline, X, y, cv=cv, groups=cv_groups, scoring=SCORING)
        for fold in range(N_SPLITS):
            for metric in SCORING:
                rows.append({
                    "scheme": scheme_name,
                    "fold":   fold + 1,
                    "metric": metric,
                    "score":  scores[f"test_{metric}"][fold],
                })
    return pd.DataFrame(rows)

In [ ]:
results_ansatz1 = []
for name, model in MODELS_ANSATZ1.items():
    df_result = evaluate_model(build_pipeline(model), X_features, y, groups)
    df_result["model"] = name
    results_ansatz1.append(df_result)

results_ansatz1 = pd.concat(results_ansatz1, ignore_index=True)
results_ansatz1.head()

### 4.3 Vergleich der Modelle

Die Ergebnisse pro Durchlauf werden für GroupKFold (realistischer Fall: Generalisierung auf eine neue, unbekannte Zeitperiode) und zum Vergleich für StratifiedKFold gezeigt. Anschließend folgt die Aggregation (Mittelwert/Std) über beide Schemata hinweg.

In [ ]:
def per_run_table(results: pd.DataFrame, scheme: str) -> pd.DataFrame:
    """Pivot per-fold F1-Macro scores (model x Durchlauf) for one CV scheme."""
    return (
        results[(results["metric"] == "f1_macro") & (results["scheme"] == scheme)]
        .pivot(index="model", columns="fold", values="score")
        .rename(columns=lambda f: f"Durchlauf {f}")
    )


print("GroupKFold (realistischer Fall: neue, unbekannte Zeitperiode)")
per_run_ansatz1_group = per_run_table(results_ansatz1, "GroupKFold")
display(per_run_ansatz1_group)

print("StratifiedKFold (zum Vergleich)")
per_run_ansatz1_stratified = per_run_table(results_ansatz1, "StratifiedKFold")
display(per_run_ansatz1_stratified)

In [ ]:
summary_ansatz1 = (
    results_ansatz1
    .groupby(["model", "scheme", "metric"])["score"]
    .agg(["mean", "std"])
    .reset_index()
)
summary_ansatz1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, metric in zip(axes, SCORING):
    data = summary_ansatz1[summary_ansatz1["metric"] == metric]
    sns.barplot(data=data, x="model", y="mean", hue="scheme", ax=ax, errorbar=None)
    for _, row in data.iterrows():
        x_pos = list(MODELS_ANSATZ1).index(row["model"])
        x_pos += -0.2 if row["scheme"] == "StratifiedKFold" else 0.2
        ax.errorbar(x_pos, row["mean"], yerr=row["std"], fmt="none", color="black", capsize=3)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=30)

axes[0].legend(title="CV-Schema")
axes[1].get_legend().remove()
plt.tight_layout()
plt.show()

### 4.4 Modellauswahl für die Konfusionsmatrix

Phase 1 hat eine zeitliche Drift dokumentiert (TS1-Korrelation mit der Aufnahmereihenfolge, r ≈ -0.90) sowie eine Aufzeichnung der Klassen in zeitlich zusammenhängenden Blöcken. Bei StratifiedKFold landen Zyklen aus demselben Block durch die zufällige Aufteilung teils gleichzeitig in Training und Test. Da sich Zyklen desselben Blocks stark ähneln, kann ein Modell dadurch block-spezifische Muster wiedererkennen, statt die eigentliche Klassentrennung zu lernen — die StratifiedKFold-Metrik wäre dann zu optimistisch.

GroupKFold hält Zyklen desselben Blocks strikt getrennt zwischen Training und Test und bildet damit den realistischeren Fall ab, dass künftige Messungen aus einer neuen, unbekannten Zeitperiode stammen. Die Modellauswahl für die Konfusionsmatrix erfolgt daher anhand des GroupKFold-F1-Macro, nicht anhand von StratifiedKFold.

Zusätzlich wird die Differenz zwischen beiden Schemata pro Modell ausgewiesen: eine große Lücke (StratifiedKFold deutlich über GroupKFold) ist ein Hinweis darauf, dass ein Modell vorrangig die Blockstruktur statt des physikalischen Klassenunterschieds lernt, und relativiert einen hohen StratifiedKFold-Wert entsprechend.

In [ ]:
overfit_gap_ansatz1 = (
    summary_ansatz1[summary_ansatz1["metric"] == "f1_macro"]
    .pivot(index="model", columns="scheme", values="mean")
    .assign(gap=lambda d: d["StratifiedKFold"] - d["GroupKFold"])
    .sort_values("gap", ascending=False)
)
overfit_gap_ansatz1

best_model_ansatz1 = (
    summary_ansatz1[(summary_ansatz1["metric"] == "f1_macro") & (summary_ansatz1["scheme"] == "GroupKFold")]
    .sort_values("mean", ascending=False)
    .iloc[0]["model"]
)
print(f"Bestes Modell (Ansatz 1, F1-Macro, GroupKFold): {best_model_ansatz1}")
print(f"StratifiedKFold-GroupKFold-Lücke dieses Modells: {overfit_gap_ansatz1.loc[best_model_ansatz1, 'gap']:.3f}")

pipeline_best = build_pipeline(MODELS_ANSATZ1[best_model_ansatz1])
y_pred = cross_val_predict(pipeline_best, X_features, y, cv=cv_group, groups=groups)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred, display_labels=CLASS_NAMES, cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title(f"Konfusionsmatrix — {best_model_ansatz1} (Out-of-Fold, GroupKFold)")
plt.tight_layout()
plt.show()

print(classification_report(
    y, y_pred, labels=list(CLASS_TO_LABEL.values()), target_names=CLASS_NAMES,
))

Im Hinblick auf die in Phase 1 identifizierte Schwierigkeit, 100 bar und 115 bar zu unterscheiden, lohnt sich ein gezielter Blick auf die Zelle "100 bar" → "115 bar" (und umgekehrt) in der Konfusionsmatrix sowie auf die entsprechenden Precision-/Recall-Werte im Classification Report.

### 4.5 Feature-Importance: Sensor und Merkmalstyp

Phase 2 hat als offenen Punkt dokumentiert, dass tsfresh die Frequenzmerkmale ohne explizite Übergabe der realen Abtastrate berechnet — betroffen sind insbesondere die 1-Hz-Sensoren (TS1–4, VS1, CE, CP, SE), deren Zeitreihen mit nur 60 Werten pro Zyklus eine deutlich geringere Frequenzauflösung als die 10-Hz-Sensoren besitzen.

Statt diese Merkmale ohne Beleg zu entfernen, wird hier anhand der RandomForest-Feature-Importance datenbasiert geprüft, ob die Frequenzmerkmale der 1-Hz-Sensoren tatsächlich auffällig wenig zur Klassifikation beitragen. Ein einzelnes, auf dem gesamten Datensatz trainiertes Modell könnte block-spezifische statt klassendiskriminierende Muster als "wichtig" einstufen — dasselbe Risiko, das bei den Klassifikationsmodellen zur StratifiedKFold-GroupKFold-Lücke führt. Die Feature-Importance wird daher je einmal auf der Trainingspartition jedes GroupKFold-Folds berechnet und über die drei Folds gemittelt: ein Feature erscheint nur dann als wichtig, wenn es über mehrere, einander unbekannte Zeitblöcke hinweg konsistent zur Klassentrennung beiträgt.

In [ ]:
def parse_feature_origin(column: str) -> tuple[str, str]:
    """Split a tsfresh column name into (sensor, domain). Domain is "frequency" or "other"."""
    sensor = column.split("__")[0]
    domain = "frequency" if any(k in column for k in FREQUENCY_KEYWORDS) else "other"
    return sensor, domain


def sensor_sampling_rate(sensor: str) -> str:
    """Map a sensor name to its native sampling rate label."""
    if sensor in SENSORS_1HZ:
        return "1 Hz"
    if sensor in SENSORS_10HZ:
        return "10 Hz"
    return "100 Hz"


def compute_groupwise_feature_importance(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    cv: GroupKFold,
) -> np.ndarray:
    """Train a RandomForest per GroupKFold training fold; return mean feature importance.

    A model fit on the full dataset could assign high importance to features that
    merely identify the temporal block (vgl. die StratifiedKFold-/GroupKFold-Lücke
    bei den Klassifikationsmodellen). Averaging importances learned only from
    within each fold's training portion avoids this and reflects information that
    generalizes across unseen time blocks.
    """
    importances_per_fold = []
    for train_idx, _ in cv.split(X, y, groups):
        fold_model = build_pipeline(
            RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE)
        )
        fold_model.fit(X.iloc[train_idx], y[train_idx])
        importances_per_fold.append(fold_model.named_steps["model"].feature_importances_)
    return np.mean(importances_per_fold, axis=0)


importances = compute_groupwise_feature_importance(X_features, y, groups, cv_group)

importance_df = pd.DataFrame({"feature": X_features.columns, "importance": importances})
importance_df[["sensor", "domain"]] = importance_df["feature"].apply(
    lambda c: pd.Series(parse_feature_origin(c))
)
importance_df["sensor_rate"] = importance_df["sensor"].apply(sensor_sampling_rate)

summary_importance = (
    importance_df
    .groupby(["sensor_rate", "domain"])["importance"]
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=summary_importance, x="sensor_rate", y="importance", hue="domain", ax=ax)
ax.set_title("Summierte Feature-Importance nach Abtastrate und Merkmalstyp")
ax.set_xlabel("Native Abtastrate der Sensorgruppe")
ax.set_ylabel("Summe RandomForest-Importance")
plt.tight_layout()
plt.show()

summary_importance

**Interpretation:** Liegt der "frequency"-Balken bei "1 Hz" deutlich niedriger als bei den anderen Abtastraten-Gruppen, bestätigt das empirisch die in Phase 2 vermutete Einschränkung der tsfresh-Frequenzmerkmale für niedrig abgetastete Sensoren — in diesem Fall können die entsprechenden Spalten vor dem finalen Training begründet entfernt werden. Zeigt sich kein auffälliger Unterschied, ist eine Entfernung nicht gerechtfertigt; die Hypothese aus Phase 2 gilt dann als durch die Daten nicht bestätigt, bleibt aber als dokumentierte Überlegung bestehen.

## 5. Datengrundlage: Preprocessierte Signale (Ansatz 2)

Ansatz 2 (Deep Learning) arbeitet direkt auf den Zeitreihen statt auf extrahierten Merkmalen. Die Preprocessing-Pipeline aus Phase 2 (Ausreißer-Clipping, Tiefpassfilter, Downsampling) wird hier repliziert, da `features_phase2.csv` nur die extrahierten Merkmale enthält, nicht die preprocessierten Rohsignale.

### 5.1 Preprocessing-Pipeline (aus Phase 2)

In [ ]:
def load_sensor(name: str) -> np.ndarray:
    """Load sensor time series. Returns array of shape (2205, timesteps)."""
    return np.loadtxt(DATA_PATH / f"{name}.txt")


def clip_outliers(data: np.ndarray, iqr_factor: float = IQR_FACTOR) -> np.ndarray:
    """Clip values outside IQR-based bounds. Returns clipped array."""
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    return np.clip(data, q1 - iqr_factor * iqr, q3 + iqr_factor * iqr)


def design_lowpass_filter(
    cutoff_hz: float,
    sampling_rate: int,
    order: int = LOWPASS_ORDER,
) -> tuple[np.ndarray, np.ndarray]:
    """Design Butterworth low-pass filter. Returns (b, a) coefficients."""
    nyquist = sampling_rate / 2
    return sig.butter(order, cutoff_hz / nyquist, btype="low", analog=False)


def apply_lowpass_filter(data: np.ndarray, cutoff_hz: float, sampling_rate: int) -> np.ndarray:
    """Apply zero-phase Butterworth low-pass filter row-wise."""
    b, a = design_lowpass_filter(cutoff_hz, sampling_rate)
    return np.apply_along_axis(lambda row: sig.filtfilt(b, a, row), axis=1, arr=data)


def downsample(data: np.ndarray, factor: int) -> np.ndarray:
    """Reduce sample count by retaining every factor-th sample."""
    return data[:, ::factor]

In [ ]:
preprocessed_sensors: dict[str, np.ndarray] = {}

for name in SENSORS_ALL:
    clipped = clip_outliers(load_sensor(name))
    if name in SENSORS_100HZ:
        filtered = apply_lowpass_filter(clipped, LOWPASS_CUTOFF_HZ, sampling_rate=100)
        preprocessed_sensors[name] = downsample(filtered, DOWNSAMPLE_FACTOR)
    else:
        preprocessed_sensors[name] = clipped

for name, arr in preprocessed_sensors.items():
    print(f"  {name:5s}: {arr.shape}")

### 5.2 Resampling auf einheitliche Zeitschritt-Länge

Nach Phase-2-Preprocessing liegen 9 Sensoren (PS1–6, EPS1, FS1, FS2) bei 600 Zeitschritten (10 Hz) und 8 Sensoren (TS1–4, VS1, CE, CP, SE) bei 60 Zeitschritten (1 Hz). Keras benötigt für CNN/LSTM einen einheitlichen 3D-Input `(Samples, Zeitschritte, Channels)`.

**Entscheidung:** Upsampling der 8 1-Hz-Sensoren auf 600 Zeitschritte via lineare Interpolation (`np.interp`), statt Downsampling der 9 10-Hz-Sensoren auf 60.

**Begründung:** Upsampling verwirft keine Information — die in Phase 2 begründete Zielrate von 10 Hz (Nutzsignal < 10 Hz) bleibt für die physikalisch relevantesten Sensoren (PS5, PS6, FS2, vgl. Phase-1-Hypothese H1) vollständig erhalten. Ein erneuter Anti-Aliasing-Filter ist beim Upsampling nicht erforderlich (nur beim Downsampling relevant).

In [ ]:
def upsample_to_length(data: np.ndarray, target_length: int) -> np.ndarray:
    """Upsample each row to target_length via linear interpolation."""
    n_cycles, n_steps = data.shape
    x_old = np.linspace(0, 1, n_steps)
    x_new = np.linspace(0, 1, target_length)
    return np.array([np.interp(x_new, x_old, row) for row in data])


signal_arrays = []
for name in SENSORS_ALL:
    arr = preprocessed_sensors[name]
    if arr.shape[1] != TARGET_TIMESTEPS:
        arr = upsample_to_length(arr, TARGET_TIMESTEPS)
    signal_arrays.append(arr)

X_signals = np.stack(signal_arrays, axis=-1)
print(f"X_signals shape: {X_signals.shape}  (Samples, Zeitschritte, Channels)")

### 5.3 Zielgrößen-Encoding für Keras

`categorical_crossentropy` erwartet One-Hot-codierte Labels.

In [ ]:
y_onehot = keras.utils.to_categorical(y, num_classes=len(CLASSES))
print(f"y_onehot shape: {y_onehot.shape}")

## 6. Ansatz 2: Deep Learning

Implementierung mit Keras auf den preprocessierten und vereinheitlichten Sensor-Zeitreihen (alle 17 Sensoren als Channels).

### 6.1 Modellauswahl & Begründung

**Baseline:**
- `MLP-Baseline` (Sequential API): einfaches Fully-Connected-Netz auf   geflattetem Input.

**Hauptmodelle (3, davon 1 CNN, 2 RNN-Varianten):**
- **1D-CNN** (Sequential API, Pflicht):
  - Drei Conv1D-Schichten mit absteigender Filteranzahl (64, 64, 32) und     absteigender Kernelgröße (7, 5, 3) — die erste Schicht erfasst breite,     grobe Muster, die folgenden Schichten verfeinern auf lokalere Strukturen.
  - ReLU-Aktivierung in allen Conv1D- und Dense-Schichten als Standardwahl     für nicht-lineare Aktivierung ohne Sättigungsproblem.
  - MaxPooling1D nach den ersten beiden Conv1D-Schichten reduziert die     Sequenzlänge und macht die gelernten Muster robuster gegenüber kleinen     zeitlichen Verschiebungen.
  - GlobalAveragePooling1D statt Flatten vor dem Dense-Kopf: hält die     Parameteranzahl gering und macht die Klassifikation unabhängig von der     exakten Position eines Musters in der Zeitreihe.
- **Bidirektionales LSTM** (Functional API, Pflicht):
  - Eine Conv1D-Schicht mit Stride 5 reduziert die Sequenzlänge von 600 auf     120 Zeitschritte vor dem LSTM — hält die Trainingszeit praktikabel, ohne     die Rohauflösung der Eingabedaten zu verändern.
  - Zwei gestapelte Bidirectional-LSTM-Schichten mit absteigender     Einheitenzahl (64, 32): die erste Schicht (`return_sequences=True`)     gibt die volle Sequenz an die zweite Schicht weiter, die zweite     komprimiert auf einen festen Ausgabevektor.
  - Bidirektionalität ist möglich, da bei der Klassifikation eines     abgeschlossenen Hydraulikzyklus die gesamte Zeitreihe bereits vorliegt     — das Modell kann damit sowohl aus vergangenen als auch zukünftigen     Zeitschritten innerhalb des Zyklus lernen.
- **GRU** (Functional API): gleicher Conv1D-Stride-5-Frontend wie das LSTM,   aber mit GRU- statt LSTM-Zellen und unidirektional. GRU-Zellen haben   keinen separaten Cell-State und damit weniger Parameter als LSTM-Zellen   — dient als leichtgewichtige, schnellere Alternative im direkten   Vergleich mit dem BiLSTM.

Dropout (0.3) nach den Dense-Schichten in allen vier Modellen als Standard-Regularisierung gegen Overfitting.

In [ ]:
def build_mlp_baseline(input_shape: tuple[int, int], n_classes: int) -> keras.Model:
    """Build a fully-connected baseline network (Sequential API)."""
    model = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ], name="mlp_baseline")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_cnn_model(input_shape: tuple[int, int], n_classes: int) -> keras.Model:
    """Build a 1D-CNN with global average pooling (Sequential API)."""
    model = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=7, activation="relu", padding="same"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, kernel_size=3, activation="relu", padding="same"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ], name="cnn_1d")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_lstm_model(input_shape: tuple[int, int], n_classes: int) -> keras.Model:
    """Build a bidirectional LSTM with a Conv1D downsampling front-end (Functional API)."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, kernel_size=5, strides=5, activation="relu", padding="same")(inputs)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name="lstm_functional")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_gru_model(input_shape: tuple[int, int], n_classes: int) -> keras.Model:
    """Build a unidirectional stacked GRU with a Conv1D downsampling front-end (Functional API)."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, kernel_size=5, strides=5, activation="relu", padding="same")(inputs)
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.GRU(32)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name="gru_functional")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


MODELS_ANSATZ2 = {
    "MLP_Baseline": build_mlp_baseline,
    "CNN_1D":       build_cnn_model,
    "LSTM_BiDir":   build_lstm_model,
    "GRU":          build_gru_model,
}

### 6.2 Standardisierung pro Sensor

**Entscheidung:** Pro Sensor (Channel) wird z-Normalisierung anhand der Trainings-Statistiken (Mittelwert, Standardabweichung) des jeweiligen CV-Folds durchgeführt — analog zu `StandardScaler`, hier für 3D-Arrays.

**Begründung:** Die 17 Sensoren haben völlig unterschiedliche physikalische Größenordnungen (Druck in bar, Temperatur in °C, Durchfluss in l/min, Vibration). Ohne Normalisierung würden große-Werte-Kanäle das Training dominieren. Die Statistiken werden ausschließlich aus den Trainingsdaten des jeweiligen Folds berechnet, um Data Leakage zu vermeiden — dies war der in Phase 2 auf Phase 3 verschobene `StandardScaler`-Punkt.

In [ ]:
def scale_signals_per_channel(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Standardize each sensor channel using training-set statistics."""
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

### 6.3 Training & Evaluation

Pro Fold werden zwei Callbacks verwendet:
- `EarlyStopping` (Monitor `val_loss`, Patience 6, beste Gewichte werden wiederhergestellt)
- `ReduceLROnPlateau` (Monitor `val_loss`, Patience 3, Faktor 0.5)

*Hinweis:* Aus Rechenzeitgründen dient der jeweilige Test-Fold zugleich als Validierungsdaten für beide Callbacks. Dies ist eine pragmatische Vereinfachung gegenüber einem zusätzlichen, dritten Validierungs-Split.

In [ ]:
def build_callbacks() -> list[keras.callbacks.Callback]:
    """Build the EarlyStopping and ReduceLROnPlateau callbacks used for every fold."""
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=KERAS_PATIENCE, restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=KERAS_LR_PATIENCE, factor=KERAS_LR_FACTOR,
        ),
    ]


def train_keras_model(
    build_fn: Callable[[tuple[int, int], int], keras.Model],
    X: np.ndarray,
    y_int: np.ndarray,
    y_oh: np.ndarray,
    cv: StratifiedKFold | GroupKFold,
    cv_groups: np.ndarray | None = None,
) -> pd.DataFrame:
    """Train and evaluate a Keras model across CV folds. Returns long-format results."""
    rows = []
    splits = cv.split(X, y_int, cv_groups) if cv_groups is not None else cv.split(X, y_int)

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train, X_test = scale_signals_per_channel(X[train_idx], X[test_idx])

        model = build_fn(X.shape[1:], len(CLASSES))
        model.fit(
            X_train, y_oh[train_idx],
            validation_data=(X_test, y_oh[test_idx]),
            epochs=KERAS_EPOCHS,
            batch_size=KERAS_BATCH_SIZE,
            callbacks=build_callbacks(),
            verbose=0,
        )

        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        rows.append({
            "fold": fold,
            "f1_macro": f1_score(
                y_int[test_idx], y_pred, average="macro",
                labels=list(CLASS_TO_LABEL.values()), zero_division=0,
            ),
            "balanced_accuracy": recall_score(
                y_int[test_idx], y_pred, average="macro",
                labels=list(CLASS_TO_LABEL.values()), zero_division=0,
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
results_ansatz2 = []
for name, build_fn in MODELS_ANSATZ2.items():
    for scheme_name, cv, cv_groups in [
        ("StratifiedKFold", cv_stratified, None),
        ("GroupKFold",      cv_group,      groups),
    ]:
        df_result = train_keras_model(build_fn, X_signals, y, y_onehot, cv, cv_groups)
        df_result_long = df_result.melt(
            id_vars="fold", value_vars=["f1_macro", "balanced_accuracy"],
            var_name="metric", value_name="score",
        )
        df_result_long["model"] = name
        df_result_long["scheme"] = scheme_name
        results_ansatz2.append(df_result_long)

results_ansatz2 = pd.concat(results_ansatz2, ignore_index=True)
results_ansatz2.head()

### 6.4 Vergleich der Modelle

Die Ergebnisse pro Durchlauf werden für GroupKFold (realistischer Fall: Generalisierung auf eine neue, unbekannte Zeitperiode) und zum Vergleich für StratifiedKFold gezeigt. Anschließend folgt die Aggregation (Mittelwert/Std) über beide Schemata hinweg.

In [ ]:
print("GroupKFold (realistischer Fall: neue, unbekannte Zeitperiode)")
per_run_ansatz2_group = per_run_table(results_ansatz2, "GroupKFold")
display(per_run_ansatz2_group)

print("StratifiedKFold (zum Vergleich)")
per_run_ansatz2_stratified = per_run_table(results_ansatz2, "StratifiedKFold")
display(per_run_ansatz2_stratified)

In [ ]:
summary_ansatz2 = (
    results_ansatz2
    .groupby(["model", "scheme", "metric"])["score"]
    .agg(["mean", "std"])
    .reset_index()
)
summary_ansatz2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, metric in zip(axes, SCORING):
    data = summary_ansatz2[summary_ansatz2["metric"] == metric]
    sns.barplot(data=data, x="model", y="mean", hue="scheme", ax=ax, errorbar=None)
    for _, row in data.iterrows():
        x_pos = list(MODELS_ANSATZ2).index(row["model"])
        x_pos += -0.2 if row["scheme"] == "StratifiedKFold" else 0.2
        ax.errorbar(x_pos, row["mean"], yerr=row["std"], fmt="none", color="black", capsize=3)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel("Score")

axes[0].legend(title="CV-Schema")
axes[1].get_legend().remove()
plt.tight_layout()
plt.show()

### 6.5 Trainingsverlauf (bestes Modell)

Die Modellauswahl erfolgt aus demselben Grund wie in Abschnitt 4.4 anhand des GroupKFold-F1-Macro statt StratifiedKFold: GroupKFold hält Zyklen desselben zeitlichen Blocks strikt getrennt zwischen Training und Test und bildet damit den realistischeren Fall künftiger, unbekannter Messperioden ab. Die Differenz zwischen beiden Schemata pro Modell wird zusätzlich ausgewiesen, um Modelle zu erkennen, die vorrangig die Blockstruktur statt des physikalischen Klassenunterschieds lernen.

Die Evaluation erfolgt zusätzlich auf dem Trainingsset, um Over- bzw. Underfitting zu beurteilen. Dafür wird das beste Modell auf dem ersten GroupKFold-Split trainiert und der Verlauf von Loss und Accuracy (Training vs. Validierung) über die Epochen visualisiert.

In [ ]:
def plot_training_history(history: keras.callbacks.History, model_name: str) -> None:
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["loss"], label="Training")
    axes[0].plot(history.history["val_loss"], label="Validierung")
    axes[0].set_title(f"Loss — {model_name}")
    axes[0].set_xlabel("Epoche")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Training")
    axes[1].plot(history.history["val_accuracy"], label="Validierung")
    axes[1].set_title(f"Accuracy — {model_name}")
    axes[1].set_xlabel("Epoche")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


overfit_gap_ansatz2 = (
    summary_ansatz2[summary_ansatz2["metric"] == "f1_macro"]
    .pivot(index="model", columns="scheme", values="mean")
    .assign(gap=lambda d: d["StratifiedKFold"] - d["GroupKFold"])
    .sort_values("gap", ascending=False)
)
overfit_gap_ansatz2

best_model_ansatz2 = (
    summary_ansatz2[(summary_ansatz2["metric"] == "f1_macro") & (summary_ansatz2["scheme"] == "GroupKFold")]
    .sort_values("mean", ascending=False)
    .iloc[0]["model"]
)
print(f"Bestes Modell (Ansatz 2, F1-Macro, GroupKFold): {best_model_ansatz2}")
print(f"StratifiedKFold-GroupKFold-Lücke dieses Modells: {overfit_gap_ansatz2.loc[best_model_ansatz2, 'gap']:.3f}")

build_fn = MODELS_ANSATZ2[best_model_ansatz2]
train_idx, test_idx = next(cv_group.split(X_signals, y, groups))
X_train, X_test = scale_signals_per_channel(X_signals[train_idx], X_signals[test_idx])

best_keras_model = build_fn(X_signals.shape[1:], len(CLASSES))
history = best_keras_model.fit(
    X_train, y_onehot[train_idx],
    validation_data=(X_test, y_onehot[test_idx]),
    epochs=KERAS_EPOCHS,
    batch_size=KERAS_BATCH_SIZE,
    callbacks=build_callbacks(),
    verbose=0,
)
plot_training_history(history, best_model_ansatz2)

### 6.6 Konfusionsmatrix & Classification Report

In [ ]:
test_loss, test_accuracy = best_keras_model.evaluate(X_test, y_onehot[test_idx], verbose=0)
print(f"Test-Loss: {test_loss:.4f}, Test-Accuracy: {test_accuracy:.4f}")

y_pred = np.argmax(best_keras_model.predict(X_test, verbose=0), axis=1)
print(classification_report(
    y[test_idx], y_pred, labels=list(CLASS_TO_LABEL.values()), target_names=CLASS_NAMES,
))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y[test_idx], y_pred, display_labels=CLASS_NAMES, cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title(f"Konfusionsmatrix — {best_model_ansatz2} (1. GroupKFold-Fold)")
plt.tight_layout()
plt.show()

Im Hinblick auf die in Phase 1 identifizierte Schwierigkeit, 100 bar und 115 bar zu unterscheiden, lohnt sich ein gezielter Blick auf die Zelle "100 bar" → "115 bar" (und umgekehrt) in der Konfusionsmatrix sowie auf die entsprechenden Precision-/Recall-Werte im Classification Report.

## 7. Gesamtvergleich Ansatz 1 vs. Ansatz 2

Beide Ansätze wurden mit identischen Metriken (F1-Macro, Balanced Accuracy) und identischen CV-Schemata (StratifiedKFold, GroupKFold) evaluiert und sind damit direkt vergleichbar.

Die Rangfolge unten basiert auf dem GroupKFold-F1-Macro, nicht auf StratifiedKFold. Ein Modell mit hohem StratifiedKFold-Wert, aber niedrigem GroupKFold-Wert hat primär die zeitliche Blockstruktur aus Phase 1 gelernt statt einer generalisierbaren Klassentrennung — ein solches Modell wäre trotz hoher StratifiedKFold-Zahl kein gutes Modell für den späteren produktiven Einsatz auf neuen, unbekannten Messperioden. Die `overfit_gap_combined`-Tabelle macht diese Lücke pro Modell explizit sichtbar.

In [ ]:
summary_combined = pd.concat([
    summary_ansatz1.assign(approach="Ansatz 1: Manual Feature Extraction"),
    summary_ansatz2.assign(approach="Ansatz 2: Deep Learning"),
], ignore_index=True)

overfit_gap_combined = (
    summary_combined[summary_combined["metric"] == "f1_macro"]
    .pivot_table(index=["approach", "model"], columns="scheme", values="mean")
    .assign(gap=lambda d: d["StratifiedKFold"] - d["GroupKFold"])
    .sort_values("gap", ascending=False)
)
overfit_gap_combined

best_overall = (
    summary_combined[(summary_combined["metric"] == "f1_macro") & (summary_combined["scheme"] == "GroupKFold")]
    .sort_values("mean", ascending=False)
)
best_overall.head(10)

**Einordnung:**

Falls Ensemble-Modelle (Ansatz 1) die Deep-Learning-Modelle (Ansatz 2) übertreffen, ist dies konsistent mit der Literatur, wonach Ensemble-Methoden auf tabellarischen Daten im Mittel besser abschneiden als neuronale Netze (Grinsztajn et al. 2022).

Drei Punkte aus Phase 1 und 2 sollten bei der Interpretation besonders berücksichtigt werden:
- **Klassenimbalance:** F1-Macro und Balanced Accuracy wurden gezielt   gewählt, um seltene Klassen nicht zu unterschätzen — ein Modell mit hoher   Accuracy, aber niedrigem F1-Macro, würde überwiegend die häufigen Klassen   korrekt vorhersagen.
- **100 bar vs. 115 bar:** Phase 1 hat diese beiden Klassen als am   schwersten trennbar identifiziert. Die Konfusionsmatrizen in Abschnitt   4.4 und 6.6 sowie die Feature-Importance-Analyse in Abschnitt 4.5 geben   Aufschluss, ob dies auch in Phase 3 bestätigt wird.
- **Blockstruktur:** GroupKFold hält Zyklen desselben Blocks strikt getrennt zwischen Training und Test, StratifiedKFold nicht. Eine deutliche Lücke zwischen beiden Ergebnissen für ein Modell zeigt, dass es block-spezifische Muster (z.B. die TS1-Drift) wiedererkennt statt die physikalische Klassentrennung zu lernen — die StratifiedKFold-Zahl ist für dieses Modell dann irreführend optimistisch. Aus diesem Grund wird die Modellauswahl in dieser Phase durchgängig anhand von GroupKFold statt StratifiedKFold getroffen.

Das/die beste(n) Modell(e) aus dieser Phase bilden die Grundlage für die Sensorreduktion in Phase 4.

Hyperparameter-Optimierung (z.B. mit Optuna) ist Gegenstand von Phase 4 (Optimierung & Sensorreduktion) und daher hier nicht durchgeführt — die hier verwendeten Hyperparameter sind bewusste, begründete Startwerte, keine optimierten Endwerte.